# Importing Libraries

In [1]:
import os
import math
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import numpy as np
import pandas as pd

# Importing Dataframe

In [2]:
airbnb = pd.read_csv('../01. Data/01. Raw Data/airbnb_paris_weekdays.csv')

In [3]:
airbnb.shape

(3130, 20)

# Cleaning the Dataframe

In [4]:
airbnb.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
0,0,296.159940,Private room,False,True,2.0,True,0,0,10.0,97.0,1,0.699821,0.193709,518.478947,25.239380,1218.662228,71.608028,2.35385,48.86282
1,1,288.237487,Private room,False,True,2.0,True,0,0,10.0,97.0,1,2.100005,0.107221,873.216962,42.507907,1000.543327,58.791463,2.32436,48.85902
2,2,211.343089,Private room,False,True,2.0,False,0,0,10.0,94.0,1,3.302325,0.234724,444.556077,21.640840,902.854467,53.051310,2.31714,48.87475
3,3,298.956100,Entire home/apt,False,False,2.0,False,0,1,9.0,91.0,1,0.547567,0.195997,542.142014,26.391291,1199.184166,70.463506,2.35600,48.86100
4,4,247.926181,Entire home/apt,False,False,4.0,False,0,0,7.0,82.0,1,1.197921,0.103573,406.928958,19.809165,1070.775497,62.918272,2.35915,48.86648


In [5]:
# Check for null values
airbnb.isnull().sum()   

Unnamed: 0                    0
realSum                       0
room_type                     0
room_shared                   0
room_private                  0
person_capacity               0
host_is_superhost             0
multi                         0
biz                           0
cleanliness_rating            0
guest_satisfaction_overall    0
bedrooms                      0
dist                          0
metro_dist                    0
attr_index                    0
attr_index_norm               0
rest_index                    0
rest_index_norm               0
lng                           0
lat                           0
dtype: int64

## DataDict

realSum: the full price of accommodation for two people and two nights in EUR

room_type: the type of the accommodation

room_shared: dummy variable for shared rooms

room_private: dummy variable for private rooms

person_capacity: the maximum number of guests

host_is_superhost: dummy variable for superhost status

multi: dummy variable if the listing belongs to hosts with 2-4 offers

biz: dummy variable if the listing belongs to hosts with more than 4 offers

cleanliness_rating: cleanliness rating

guest_satisfaction_overall: overall rating of the listing

bedrooms: number of bedrooms (0 for studios)

dist: distance from city centre in km

metro_dist: distance from nearest metro station in km

attr_index: attraction index of the listing location

attr_index_norm: normalised attraction index (0-100)

rest_index: restaurant index of the listing location

attr_index_norm: normalised restaurant index (0-100)

lng: longitude of the listing location

lat: latitude of the listing location

city: city of the listing location

weekend: data for weekend or weekday. Now it's string variable, let's convert it to dummy variable: 1 for weekend, o for weekday

In [6]:
# dropping unecessary columns
airbnb.drop(columns=['Unnamed: 0', 'multi', 'biz', 'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index', 'rest_index_norm'], inplace=True)

In [7]:
# checking if dropping worked
airbnb.head()

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,lng,lat
0,296.159940,Private room,False,True,2.0,True,10.0,97.0,1,0.699821,2.35385,48.86282
1,288.237487,Private room,False,True,2.0,True,10.0,97.0,1,2.100005,2.32436,48.85902
2,211.343089,Private room,False,True,2.0,False,10.0,94.0,1,3.302325,2.31714,48.87475
3,298.956100,Entire home/apt,False,False,2.0,False,9.0,91.0,1,0.547567,2.35600,48.86100
4,247.926181,Entire home/apt,False,False,4.0,False,7.0,82.0,1,1.197921,2.35915,48.86648


In [8]:
# definition of a function to check the unique values in each column

def unique_counts_col(df):
    for col in df.columns:
        print(f'Column: {col}')
        print(df[col].value_counts())
        print('-----------------------------')

unique_counts_col(airbnb)

Column: realSum
realSum
278.217914     39
359.073539     33
336.005219     30
370.724205     30
243.265915     26
               ..
185.944636      1
4184.686364     1
412.666605      1
213.673222      1
136.079784      1
Name: count, Length: 1178, dtype: int64
-----------------------------
Column: room_type
room_type
Entire home/apt    2325
Private room        758
Shared room          47
Name: count, dtype: int64
-----------------------------
Column: room_shared
room_shared
False    3083
True       47
Name: count, dtype: int64
-----------------------------
Column: room_private
room_private
False    2372
True      758
Name: count, dtype: int64
-----------------------------
Column: person_capacity
person_capacity
2.0    1746
4.0     775
3.0     302
6.0     192
5.0     115
Name: count, dtype: int64
-----------------------------
Column: host_is_superhost
host_is_superhost
False    2701
True      429
Name: count, dtype: int64
-----------------------------
Column: cleanliness_rating
cleanli

In [9]:
# dropping more columns after checking the unique values
airbnb.drop(columns=['room_shared', 'room_private'], inplace=True)

In [10]:
# renaming columns for better understanding
airbnb.rename(columns={'realSum': 'cost_2pax_2night', 'lat': 'latitude', 'lng': 'longitude', 'dist': 'distance_km'}, inplace=True)

In [11]:
# checking further dropping and renaming
airbnb.head()

,cost_2pax_2night,room_type,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,bedrooms,distance_km,longitude,latitude
0,296.159940,Private room,2.0,True,10.0,97.0,1,0.699821,2.35385,48.86282
1,288.237487,Private room,2.0,True,10.0,97.0,1,2.100005,2.32436,48.85902
2,211.343089,Private room,2.0,False,10.0,94.0,1,3.302325,2.31714,48.87475
3,298.956100,Entire home/apt,2.0,False,9.0,91.0,1,0.547567,2.35600,48.86100
4,247.926181,Entire home/apt,4.0,False,7.0,82.0,1,1.197921,2.35915,48.86648


In [12]:
# creating a new column to differentiate between weekday and weekend listings (for when dfs are merged later on)
airbnb['type_of_day'] = 'weekday'


In [13]:
# creating a new column with the cost per day for 2 pax
airbnb['cost_2pax_1night'] = airbnb['cost_2pax_2night'] / 2

In [14]:
# checking column creation
airbnb.head()

,cost_2pax_2night,room_type,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,bedrooms,distance_km,longitude,latitude,type_of_day,cost_2pax_1night
0,296.159940,Private room,2.0,True,10.0,97.0,1,0.699821,2.35385,48.86282,weekday,148.079970
1,288.237487,Private room,2.0,True,10.0,97.0,1,2.100005,2.32436,48.85902,weekday,144.118744
2,211.343089,Private room,2.0,False,10.0,94.0,1,3.302325,2.31714,48.87475,weekday,105.671544
3,298.956100,Entire home/apt,2.0,False,9.0,91.0,1,0.547567,2.35600,48.86100,weekday,149.478050
4,247.926181,Entire home/apt,4.0,False,7.0,82.0,1,1.197921,2.35915,48.86648,weekday,123.963091


In [15]:
# reordering columns for better understanding
airbnb = airbnb[['type_of_day', 'cost_2pax_2night', 'cost_2pax_1night', 'room_type', 'bedrooms', 'person_capacity', 'host_is_superhost', 'cleanliness_rating', 'guest_satisfaction_overall', 'longitude', 'latitude', 'distance_km']]

In [16]:
# checking the final dataframe
airbnb.head()

,type_of_day,cost_2pax_2night,cost_2pax_1night,room_type,bedrooms,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,longitude,latitude,distance_km
0,weekday,296.159940,148.079970,Private room,1,2.0,True,10.0,97.0,2.35385,48.86282,0.699821
1,weekday,288.237487,144.118744,Private room,1,2.0,True,10.0,97.0,2.32436,48.85902,2.100005
2,weekday,211.343089,105.671544,Private room,1,2.0,False,10.0,94.0,2.31714,48.87475,3.302325
3,weekday,298.956100,149.478050,Entire home/apt,1,2.0,False,9.0,91.0,2.35600,48.86100,0.547567
4,weekday,247.926181,123.963091,Entire home/apt,1,4.0,False,7.0,82.0,2.35915,48.86648,1.197921


In [17]:
# exporting the cleaned dataframe
airbnb.to_pickle('../01. Data/02. Clean Data/airbnb_paris_weekdays_cleaned.pkl')